# Description

## This notebook is used to extract the VCs from GPs linkedin connections

In [1]:
import logging

import ck_marketing.linkedin.profile_filtering as cmliprfi
from ck_marketing.hunterio.hunterapi import GoogleSheetsHelper
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint

In [2]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

DEBUG:helpers.hsystem:> (cd . && cd "$(git rev-parse --show-toplevel)/.." && (git rev-parse --is-inside-work-tree | grep -q true)) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --show-toplevel) 2>&1
DEBUG:helpers.hsystem:> (git branch --show-current) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --short HEAD) 2>&1
DEBUG:helpers.hsystem:> (git log --date=local --oneline --graph --date-order --decorate --pretty=format:'%h %<(8)%aN%  %<(65)%s (%>(14)%ar) %ad %<(10)%d' -3) 2>&1


INFO:__main__:# Git
  branch_name='CmampTask8908_Find_GP_VC_connections'
  hash='558cafea5'
  # Last commits:
    *   558cafea5 shaunak01 Merge branch 'master' into CmampTask8908_Find_GP_VC_connections   ( 7 minutes ago) Mon Jul 15 14:22:23 2024  (HEAD -> CmampTask8908_Find_GP_VC_connections)
    |\  
    | * 936bd502e Juraj Smeriga CmampTask8680_Group_HW_resource_usage_of_ECS_tasks_by_type_of_workload (#9033) (   5 hours ago) Mon Jul 15 09:17:08 2024  (origin/master, origin/HEAD, origin/CmampTask9005_Do_not_print_AWS_credentials_in_plaintext_in_helpers.haws.get_session, master)
    | * a3000a27b pavolrabatin CmTask8737_Modify_RDS_Terraform_module_with_dynamic_per_instance_DB_username_and_password_variables (#8936) (    2 days ago) Sat Jul 13 18:59:32 2024           
# Machine info
  system=Linux
  node name=d001f97bed69
  release=5.15.0-1056-aws
  version=#61~20.04.1-Ubuntu SMP Wed Mar 13 17:40:41 UTC 2024
  machine=x86_64
  processor=x86_64
  cpu count=8
  cpu freq=scpufreq(current=2

In [20]:
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)

In [21]:
df = google_sheet_helper.read_sheet(
    "1oJ8wHXWDBeyVW4WQt08oC3X3Nl7HCnqClalSvOj5wNs"
)

In [22]:
words = ["Partner", "VC", "invest", "Venture", "Director"]

In [23]:
filtered_df_VC = cmliprfi.filter_df(df, "title", words, "keep")
filtered_df_non_VC = cmliprfi.filter_df(df, "title", words, "remove")

INFO:ck_marketing.linkedin.profile_filtering:Filtered dataframe to keep rows where 'title' contains any of ['Partner', 'VC', 'invest', 'Venture', 'Director'].
INFO:ck_marketing.linkedin.profile_filtering:609 entries were kept.
INFO:ck_marketing.linkedin.profile_filtering:Entries before filter: 1500, Entries after filter: 609
INFO:ck_marketing.linkedin.profile_filtering:Original entries: 1500
INFO:ck_marketing.linkedin.profile_filtering:Remaining entries after filtering: 609
INFO:ck_marketing.linkedin.profile_filtering:Removed entries: 891
INFO:ck_marketing.linkedin.profile_filtering:Percentage of entries removed: 59.40%
INFO:ck_marketing.linkedin.profile_filtering:Filtered dataframe to remove rows where 'title' contains any of ['Partner', 'VC', 'invest', 'Venture', 'Director'].
INFO:ck_marketing.linkedin.profile_filtering:609 entries were removed.
INFO:ck_marketing.linkedin.profile_filtering:Entries before filter: 1500, Entries after filter: 891
INFO:ck_marketing.linkedin.profile_filte

In [24]:
file_id = "1oJ8wHXWDBeyVW4WQt08oC3X3Nl7HCnqClalSvOj5wNs"
sheet = google_sheet_helper.google_account.open_by_key(file_id)
sheet.add_worksheet(title="VC", rows="100", cols="20")
sheet.add_worksheet(title="non_VC", rows="100", cols="20")
google_sheet_helper.write_results(file_id, filtered_df_VC, "VC")
google_sheet_helper.write_results(file_id, filtered_df_non_VC, "non_VC")
print(
    f"Filtered DataFrames written in Google Sheet with file ID '{file_id}' successfully."
)

INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: VC
INFO:ck_marketing.hunterio.hunterapi:Email extraction completed. Results saved in the new tab: non_VC


Filtered DataFrames written in Google Sheet with file ID '1oJ8wHXWDBeyVW4WQt08oC3X3Nl7HCnqClalSvOj5wNs' successfully.


## Some Stats

In [28]:
df_clean = google_sheet_helper.read_sheet(
    "1oJ8wHXWDBeyVW4WQt08oC3X3Nl7HCnqClalSvOj5wNs", "cleaned_profiles"
)

In [30]:
# Count the total number of 'FP' and 'FN'
fp_count = df_clean[df_clean["Predicted"] == "FP"].shape[0]
fn_count = df_clean[df_clean["Predicted"] == "FN"].shape[0]
tp_count = df_clean[df_clean["Predicted"] == "TP"].shape[0]
tn_count = df_clean[df_clean["Predicted"] == "TN"].shape[0]


print(f"Total number of FP: {fp_count}")
print(f"Total number of FN: {fn_count}")
print(f"Total number of FP: {tp_count}")
print(f"Total number of FN: {tn_count}")

Total number of FP: 5
Total number of FN: 2
Total number of FP: 41
Total number of FN: 15


In [31]:
# Filter rows with 'FP' and 'FN'
fp_rows = df_clean[df_clean["Predicted"] == "FP"][["fullName", "title"]]
fn_rows = df_clean[df_clean["Predicted"] == "FN"][["fullName", "title"]]

print("Rows with FP:")
print(fp_rows)

print("Rows with FN:")
print(fn_rows)

Rows with FP:
                 fullName                                              title
38               Steve Fu  Seeking innovative startups driving applicatio...
49           Paola Origel  Co-founder and CEO | Investor | Mentor | Speak...
56             Lucas Wang  Business Development & Investment in the Susta...
59  Francis X. Frecentese  Managing Director, Strategic Investments at Ba...
62           Alan Quigley                                Investor | Attorney
Rows with FN:
           fullName                                              title
6    Precious Worgu  Recent Data Science and Analytics Graduate fro...
44  Alireza Khaligh  Professor & Director @ UMD. President @ AmpX E...
